# 6. LSTM Modeling

5번 노트북에서 생성한 LSTM tensor를 사용해 4개 12시간 time step 입력으로 3개 horizon target을 동시에 예측합니다.

- Input: `t-3`, `t-2`, `t-1`, `t`
- Output: `y_t`, `y_t_plus_1`, `y_t_plus_2`
- Tuning 기준: horizon별 AUPRC의 macro average

In [ ]:
from pathlib import Path
import json
import random
from itertools import product

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import average_precision_score, confusion_matrix, roc_auc_score
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

RANDOM_STATE = 42
VAL_SIZE = 0.20
MAX_EPOCHS = 25
PATIENCE = 5
HORIZONS = ["y_t", "y_t_plus_1", "y_t_plus_2"]

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)
torch.set_num_threads(2)

SRC_DIR = Path.cwd().resolve()
PROJECT_DIR = SRC_DIR.parent
MODELING_DIR = PROJECT_DIR / "processed" / "modeling"
MODEL_DIR = PROJECT_DIR / "models"
OUTPUT_DIR = PROJECT_DIR / "outputs" / "modeling"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("device:", device)
print(f"MODELING_DIR: {MODELING_DIR}")

In [ ]:
required_files = {
    "X_train": MODELING_DIR / "X_train_lstm.npy",
    "X_test": MODELING_DIR / "X_test_lstm.npy",
    "y_train_any": MODELING_DIR / "y_train_lstm.npy",
    "y_test_any": MODELING_DIR / "y_test_lstm.npy",
    "y_train_steps": MODELING_DIR / "y_train_steps_lstm.npy",
    "y_test_steps": MODELING_DIR / "y_test_steps_lstm.npy",
    "meta_train": MODELING_DIR / "lstm_train_metadata.csv",
    "meta_test": MODELING_DIR / "lstm_test_metadata.csv",
}
missing = [str(path) for path in required_files.values() if not path.exists()]
if missing:
    raise FileNotFoundError("Missing preprocessing outputs:\n" + "\n".join(missing))

X_train = np.load(required_files["X_train"]).astype(np.float32)
X_test = np.load(required_files["X_test"]).astype(np.float32)
y_train_any = np.load(required_files["y_train_any"]).astype(np.float32)
y_test_any = np.load(required_files["y_test_any"]).astype(np.float32)
y_train_steps = np.load(required_files["y_train_steps"]).astype(np.float32)
y_test_steps = np.load(required_files["y_test_steps"]).astype(np.float32)
meta_train = pd.read_csv(required_files["meta_train"])
meta_test = pd.read_csv(required_files["meta_test"])

print("X_train", X_train.shape, "any positive rate", y_train_any.mean())
print("X_test", X_test.shape, "any positive rate", y_test_any.mean())
print("train horizon positive rates", dict(zip(HORIZONS, y_train_steps.mean(axis=0))))
print("test horizon positive rates", dict(zip(HORIZONS, y_test_steps.mean(axis=0))))
print("NaN count", np.isnan(X_train).sum(), np.isnan(X_test).sum())

## Subject-level validation split

In [ ]:
def make_validation_subjects(meta: pd.DataFrame, y_any: np.ndarray, val_size: float, random_state: int) -> set:
    rng = np.random.default_rng(random_state)
    subject_summary = meta.assign(y_any=y_any).groupby("subject_id", as_index=False)["y_any"].max()
    val_subjects = []
    class_counts = subject_summary["y_any"].value_counts()
    can_stratify = subject_summary["y_any"].nunique() > 1 and class_counts.min() >= 2
    if can_stratify:
        for _, group in subject_summary.groupby("y_any"):
            subjects = group["subject_id"].to_numpy()
            n_val = int(round(len(subjects) * val_size))
            n_val = min(max(n_val, 1), len(subjects) - 1)
            val_subjects.extend(rng.choice(subjects, size=n_val, replace=False).tolist())
    else:
        subjects = subject_summary["subject_id"].to_numpy()
        n_val = int(round(len(subjects) * val_size))
        n_val = min(max(n_val, 1), len(subjects) - 1)
        val_subjects = rng.choice(subjects, size=n_val, replace=False).tolist()
    return set(val_subjects)


val_subjects = make_validation_subjects(meta_train, y_train_any, VAL_SIZE, RANDOM_STATE)
val_mask = meta_train["subject_id"].isin(val_subjects).to_numpy()
inner_train_mask = ~val_mask

X_inner = X_train[inner_train_mask]
X_val = X_train[val_mask]
y_inner_steps = y_train_steps[inner_train_mask]
y_val_steps = y_train_steps[val_mask]
y_inner_any = y_train_any[inner_train_mask]
y_val_any = y_train_any[val_mask]
meta_inner = meta_train.loc[inner_train_mask].reset_index(drop=True)
meta_val = meta_train.loc[val_mask].reset_index(drop=True)

print("inner train", X_inner.shape, "any positive", y_inner_any.mean(), "subjects", meta_inner["subject_id"].nunique())
print("validation", X_val.shape, "any positive", y_val_any.mean(), "subjects", meta_val["subject_id"].nunique())
print("validation horizon positive rates", dict(zip(HORIZONS, y_val_steps.mean(axis=0))))
print("subject overlap", len(set(meta_inner["subject_id"]) & set(meta_val["subject_id"])))

## Multi-output LSTM model과 평가 함수

In [ ]:
class LSTMMultiOutputClassifier(nn.Module):
    def __init__(self, input_size: int, hidden_size: int, num_layers: int, dropout: float, output_size: int = 3):
        super().__init__()
        lstm_dropout = dropout if num_layers > 1 else 0.0
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=lstm_dropout,
        )
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        _, (hidden, _) = self.lstm(x)
        last_hidden = hidden[-1]
        return self.classifier(self.dropout(last_hidden))


def make_loader(x: np.ndarray, y: np.ndarray, batch_size: int, shuffle: bool) -> DataLoader:
    dataset = TensorDataset(torch.from_numpy(x), torch.from_numpy(y))
    generator = torch.Generator().manual_seed(RANDOM_STATE)
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, generator=generator, num_workers=0)


def binary_metric_dict(y_true: np.ndarray, y_prob: np.ndarray, threshold: float = 0.5) -> dict:
    y_true = y_true.astype(int)
    y_pred = (y_prob >= threshold).astype(int)
    if len(np.unique(y_true)) == 2:
        auroc = roc_auc_score(y_true, y_prob)
        auprc = average_precision_score(y_true, y_prob)
    else:
        auroc = np.nan
        auprc = np.nan
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    sensitivity = tp / (tp + fn) if (tp + fn) else np.nan
    specificity = tn / (tn + fp) if (tn + fp) else np.nan
    ppv = tp / (tp + fp) if (tp + fp) else np.nan
    npv = tn / (tn + fn) if (tn + fn) else np.nan
    return {
        "auroc": auroc,
        "auprc": auprc,
        "sensitivity": sensitivity,
        "specificity": specificity,
        "ppv": ppv,
        "npv": npv,
        "tp": int(tp), "fp": int(fp), "tn": int(tn), "fn": int(fn),
    }


def multioutput_metrics(y_true: np.ndarray, y_prob: np.ndarray) -> tuple[pd.DataFrame, dict]:
    rows = []
    for idx, horizon in enumerate(HORIZONS):
        row = {"horizon": horizon, **binary_metric_dict(y_true[:, idx], y_prob[:, idx])}
        rows.append(row)
    metrics_df = pd.DataFrame(rows)

    y_true_any = y_true.max(axis=1)
    y_prob_any = 1.0 - np.prod(1.0 - y_prob, axis=1)
    any_metrics = binary_metric_dict(y_true_any, y_prob_any)
    any_metrics = {f"any_{k}": v for k, v in any_metrics.items()}

    summary = {
        "macro_auroc": float(metrics_df["auroc"].mean()),
        "macro_auprc": float(metrics_df["auprc"].mean()),
        **any_metrics,
    }
    for _, row in metrics_df.iterrows():
        summary[f"{row['horizon']}_auroc"] = row["auroc"]
        summary[f"{row['horizon']}_auprc"] = row["auprc"]
    return metrics_df, summary


def predict_proba(model: nn.Module, x: np.ndarray, batch_size: int = 256) -> np.ndarray:
    model.eval()
    loader = make_loader(x, np.zeros((len(x), len(HORIZONS)), dtype=np.float32), batch_size=batch_size, shuffle=False)
    probs = []
    with torch.no_grad():
        for xb, _ in loader:
            xb = xb.to(device)
            logits = model(xb)
            probs.append(torch.sigmoid(logits).detach().cpu().numpy())
    return np.concatenate(probs, axis=0)

## Hyperparameter tuning

In [ ]:
def train_trial(params: dict) -> tuple[dict, dict]:
    model = LSTMMultiOutputClassifier(
        input_size=X_train.shape[2],
        hidden_size=params["hidden_size"],
        num_layers=params["num_layers"],
        dropout=params["dropout"],
        output_size=len(HORIZONS),
    ).to(device)

    n_pos = y_inner_steps.sum(axis=0)
    n_neg = y_inner_steps.shape[0] - n_pos
    pos_weight = torch.tensor(n_neg / np.maximum(n_pos, 1.0), dtype=torch.float32, device=device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = torch.optim.AdamW(model.parameters(), lr=params["lr"], weight_decay=params["weight_decay"])
    train_loader = make_loader(X_inner, y_inner_steps, params["batch_size"], shuffle=True)

    best_state = None
    best_score = -np.inf
    best_epoch = 0
    epochs_without_improve = 0
    history = []

    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        losses = []
        for xb, yb in train_loader:
            xb = xb.to(device)
            yb = yb.to(device)
            optimizer.zero_grad(set_to_none=True)
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            optimizer.step()
            losses.append(float(loss.detach().cpu()))

        val_prob = predict_proba(model, X_val)
        _, val_summary = multioutput_metrics(y_val_steps, val_prob)
        score = val_summary["macro_auprc"] if not np.isnan(val_summary["macro_auprc"]) else -np.inf
        history.append({"epoch": epoch, "train_loss": float(np.mean(losses)), **val_summary})

        if score > best_score:
            best_score = score
            best_epoch = epoch
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            epochs_without_improve = 0
        else:
            epochs_without_improve += 1
            if epochs_without_improve >= PATIENCE:
                break

    model.load_state_dict(best_state)
    val_prob = predict_proba(model, X_val)
    val_horizon_metrics, final_summary = multioutput_metrics(y_val_steps, val_prob)
    final_summary.update({"best_epoch": best_epoch, "epochs_run": len(history), "best_score": best_score})
    artifact = {"state_dict": best_state, "history": history, "params": params, "val_horizon_metrics": val_horizon_metrics}
    return final_summary, artifact


param_grid = [
    {
        "hidden_size": hidden_size,
        "num_layers": num_layers,
        "dropout": dropout,
        "lr": lr,
        "batch_size": 64,
        "weight_decay": 1e-4,
    }
    for hidden_size, num_layers, dropout, lr in product([32, 64], [1, 2], [0.0, 0.2], [1e-3])
]

print("n_trials", len(param_grid))

In [ ]:
trial_rows = []
best_artifact = None
best_trial_score = -np.inf

for trial_id, params in enumerate(param_grid, start=1):
    print(f"Trial {trial_id}/{len(param_grid)}: {params}")
    metrics, artifact = train_trial(params)
    row = {"trial_id": trial_id, **params, **metrics}
    trial_rows.append(row)
    print({k: row[k] for k in ["macro_auprc", "macro_auroc", "any_auprc", "any_auroc", "best_epoch", "epochs_run"]})
    if metrics["macro_auprc"] > best_trial_score:
        best_trial_score = metrics["macro_auprc"]
        best_artifact = artifact | {"trial_id": trial_id, "metrics": metrics}

tuning_results = pd.DataFrame(trial_rows).sort_values(["macro_auprc", "macro_auroc"], ascending=False).reset_index(drop=True)
display(tuning_results)
tuning_results.to_csv(OUTPUT_DIR / "lstm_tuning_results.csv", index=False)

## Best model test 평가

In [ ]:
best_params = best_artifact["params"]
best_model = LSTMMultiOutputClassifier(
    input_size=X_train.shape[2],
    hidden_size=best_params["hidden_size"],
    num_layers=best_params["num_layers"],
    dropout=best_params["dropout"],
    output_size=len(HORIZONS),
).to(device)
best_model.load_state_dict(best_artifact["state_dict"])

test_prob = predict_proba(best_model, X_test)
test_horizon_metrics, test_summary = multioutput_metrics(y_test_steps, test_prob)
test_summary.update({"split": "test", "target": "multioutput_y_t_to_y_t_plus_2", **best_params})
test_horizon_metrics.insert(0, "split", "test")

test_predictions = meta_test.copy()
for idx, horizon in enumerate(HORIZONS):
    test_predictions[f"{horizon}_true"] = y_test_steps[:, idx].astype(int)
    test_predictions[f"{horizon}_prob"] = test_prob[:, idx]
    test_predictions[f"{horizon}_pred_0_5"] = (test_prob[:, idx] >= 0.5).astype(int)
test_predictions["y_any_true"] = y_test_steps.max(axis=1).astype(int)
test_predictions["y_any_prob_from_horizons"] = 1.0 - np.prod(1.0 - test_prob, axis=1)

pd.DataFrame([test_summary]).to_csv(OUTPUT_DIR / "lstm_test_metrics.csv", index=False)
test_horizon_metrics.to_csv(OUTPUT_DIR / "lstm_test_metrics_by_horizon.csv", index=False)
test_predictions.to_csv(OUTPUT_DIR / "lstm_test_predictions.csv", index=False)

model_payload = {
    "model_state_dict": best_artifact["state_dict"],
    "params": best_params,
    "validation_metrics": best_artifact["metrics"],
    "validation_metrics_by_horizon": best_artifact["val_horizon_metrics"].to_dict(orient="records"),
    "test_metrics": test_summary,
    "test_metrics_by_horizon": test_horizon_metrics.to_dict(orient="records"),
    "input_size": int(X_train.shape[2]),
    "sequence_length": int(X_train.shape[1]),
    "output_size": len(HORIZONS),
    "horizons": HORIZONS,
    "target": "multioutput_y_t_to_y_t_plus_2",
}
torch.save(model_payload, MODEL_DIR / "lstm_best_model.pt")

with open(MODEL_DIR / "lstm_best_model_config.json", "w") as f:
    json.dump(
        {
            "params": best_params,
            "validation_metrics": best_artifact["metrics"],
            "validation_metrics_by_horizon": best_artifact["val_horizon_metrics"].to_dict(orient="records"),
            "test_metrics": test_summary,
            "test_metrics_by_horizon": test_horizon_metrics.to_dict(orient="records"),
            "input_size": int(X_train.shape[2]),
            "sequence_length": int(X_train.shape[1]),
            "output_size": len(HORIZONS),
            "horizons": HORIZONS,
            "target": "multioutput_y_t_to_y_t_plus_2",
        },
        f,
        indent=2,
    )

display(pd.DataFrame([test_summary]))
display(test_horizon_metrics)
print("Saved multi-output best model and evaluation outputs")